# 04b. Head-Weight Estimation from Recommended-vs-Unrecommended Pairs

This notebook estimates the four scalar head weights using sampled validation pairs from notebook 04a. The target is behavioral matching:

$$
    \text{real recommended videos should score above sampled unrecommended videos.}
$$

For pair $i$, let $j_i^+$ be the recommended video and $j_i^-$ be a sampled unrecommended video in the same user/session context. The frozen head model gives

$$
    \hat p_{i}^{+}
    =
    (\hat p^{complete}_{i+},\hat p^{long}_{i+},\hat p^{rewatch}_{i+},\hat p^{neg}_{i+}),
$$

and

$$
    \hat p_{i}^{-}
    =
    (\hat p^{complete}_{i-},\hat p^{long}_{i-},\hat p^{rewatch}_{i-},\hat p^{neg}_{i-}).
$$

The score margin is

$$
    m_i(w)
    =
    w^\top(\hat p_i^+ - \hat p_i^-).
$$

## Identification Caveat for Downstream Use

These head-weight estimates are useful diagnostics, but they are not a clean structural utility scale.

- The observed-order Plackett-Luce fit barely improves over random ranking: the saved diagnostics report PL NLL around `1.664017` for the unconstrained fit versus `1.664552` for random ranking. The unconstrained PL fit also assigns a negative rewatch weight, so the signs are not fully behaviorally stable.
- The recommended-vs-sampled-unrecommended pairwise objective is more problematic because sampled unrecommended videos are not the true platform candidate set. In the saved diagnostics, the unconstrained pairwise fit gives negative weights to completion, long-watch, and rewatch, which means unrecommended sampled videos often have better predicted behavior scores than recommended videos under those heads.

For structural estimation, notebook 06 therefore uses the transparent naive score proxy rather than treating either head-weight fit as the final utility measure.


## Objective

The main estimator is Bradley-Terry pairwise logistic regression:

$$
    \min_w
    \frac{1}{\sum_i a_i}
    \sum_i
    a_i\log\left(1+\exp\left[-\frac{w^\top\Delta \hat p_i}{\tau}\right]\right)
    +
    \lambda \lVert w\rVert_2^2,
$$

where

$$
    \Delta \hat p_i = \hat p_i^+ - \hat p_i^-.
$$

The constrained fit imposes the economically natural sign restrictions

$$
    w_{complete}\ge 0,
    \qquad
    w_{long}\ge 0,
    \qquad
    w_{rewatch}\ge 0,
    \qquad
    w_{neg}\le 0.
$$

This is a convex optimization problem because the logistic loss is convex in the linear margin and the constraints are linear box constraints.


In [ ]:
from pathlib import Path
from dataclasses import dataclass
import json

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.optimize import minimize
from scipy.special import expit
from sklearn.metrics import roc_auc_score, log_loss

PROJECT_ROOT = Path('/Users/haozhangao/Desktop/RecSys Research')
NEW_CODE = PROJECT_ROOT / 'python_code_new'
OUTPUT_DIR = NEW_CODE / 'outputs'
DOCS_DIR = NEW_CODE / 'docs'

PAIR_DELTAS_PATH = OUTPUT_DIR / 'validation_pair_head_deltas.parquet'
CANDIDATE_DIAGNOSTICS_PATH = OUTPUT_DIR / 'validation_candidate_generation_diagnostics.json'
WEIGHTS_PATH = OUTPUT_DIR / 'head_weight_pairwise_weights.json'
METRICS_PATH = OUTPUT_DIR / 'head_weight_pairwise_metrics_validation.json'
SCORES_PATH = OUTPUT_DIR / 'validation_pair_scores_with_head_weights.parquet'
REPORT_PATH = DOCS_DIR / 'head_weight_pairwise_report.md'

DELTA_COLS = ['delta_complete', 'delta_long', 'delta_rewatch', 'delta_neg']
WEIGHT_NAMES = ['w_complete', 'w_long', 'w_rewatch', 'w_neg']
HEAD_COLS = ['p_complete', 'p_long', 'p_rewatch', 'p_neg']

TAU = 1.0
LAMBDA_L2 = 1e-4
PRIMARY_FIT = 'constrained'  # choices: 'constrained', 'unconstrained'
PAIR_WEIGHTING = 'pair'  # choices: 'pair', 'positive', 'session'
WRITE_OUTPUTS = True

if PRIMARY_FIT not in {'constrained', 'unconstrained'}:
    raise ValueError("PRIMARY_FIT must be 'constrained' or 'unconstrained'")
if PAIR_WEIGHTING not in {'pair', 'positive', 'session'}:
    raise ValueError("PAIR_WEIGHTING must be 'pair', 'positive', or 'session'")

DOCS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('pair deltas path:', PAIR_DELTAS_PATH)
print('primary fit:', PRIMARY_FIT)
print('pair weighting:', PAIR_WEIGHTING)


## 1. Load Pair Deltas

The input file should come from notebook 04a and contain one row per sampled pair, including the four head differences. If an older artifact is loaded, missing diagnostic fields are filled with `unknown` so the estimator can still run, but the report will make that clear.


In [ ]:
pairs = pd.read_parquet(PAIR_DELTAS_PATH).copy()
missing_cols = sorted(set(DELTA_COLS) - set(pairs.columns))
if missing_cols:
    raise ValueError(f'Missing required delta columns: {missing_cols}')

if 'negative_type' not in pairs.columns:
    pairs['negative_type'] = 'unknown_old_artifact'
if 'negative_exclusion_policy' not in pairs.columns:
    pairs['negative_exclusion_policy'] = 'unknown_old_artifact'
if 'negative_prior_seen_by_user' not in pairs.columns:
    pairs['negative_prior_seen_by_user'] = np.nan
if 'negative_future_seen_by_user' not in pairs.columns:
    pairs['negative_future_seen_by_user'] = np.nan
if 'negative_ever_seen_by_user' not in pairs.columns:
    pairs['negative_ever_seen_by_user'] = np.nan

if 'split' in pairs.columns and not pairs['split'].eq('val').all():
    bad = pairs.loc[~pairs['split'].eq('val'), 'split'].value_counts(dropna=False)
    raise ValueError(f'Non-validation pairs appeared: {bad.to_dict()}')

if pairs[DELTA_COLS].isna().any().any():
    raise ValueError('Pair delta table has missing delta cells')

sample_summary = {
    'pairs': int(len(pairs)),
    'sessions': int(pairs['session_key'].nunique()),
    'users': int(pairs['user_id'].nunique()),
    'positive_session_videos': int(pairs[['session_key', 'j_plus']].drop_duplicates().shape[0]),
    'negative_session_videos': int(pairs[['session_key', 'j_minus']].drop_duplicates().shape[0]),
    'negative_types': pairs['negative_type'].value_counts(dropna=False).to_dict(),
    'negative_exclusion_policies': pairs['negative_exclusion_policy'].value_counts(dropna=False).to_dict(),
    'negative_prior_seen_by_user_sum': None if pairs['negative_prior_seen_by_user'].isna().all() else int(pairs['negative_prior_seen_by_user'].fillna(0).sum()),
    'negative_future_seen_by_user_sum': None if pairs['negative_future_seen_by_user'].isna().all() else int(pairs['negative_future_seen_by_user'].fillna(0).sum()),
    'negative_ever_seen_by_user_sum': None if pairs['negative_ever_seen_by_user'].isna().all() else int(pairs['negative_ever_seen_by_user'].fillna(0).sum()),
}
print(json.dumps(sample_summary, indent=2, default=str))
display(pairs[DELTA_COLS + ['negative_type', 'negative_exclusion_policy']].describe(include='all').T)


In [ ]:
def make_sample_weights(frame, weighting):
    if weighting == 'pair':
        return np.ones(len(frame), dtype=np.float64)
    if weighting == 'positive':
        group_size = frame.groupby(['session_key', 'j_plus'], observed=True)['pair_id'].transform('size')
        return (1.0 / group_size.to_numpy(dtype=np.float64))
    if weighting == 'session':
        group_size = frame.groupby('session_key', observed=True)['pair_id'].transform('size')
        return (1.0 / group_size.to_numpy(dtype=np.float64))
    raise ValueError(weighting)

X = pairs[DELTA_COLS].to_numpy(dtype=np.float64)
sample_weight = make_sample_weights(pairs, PAIR_WEIGHTING)
sample_weight = sample_weight / np.mean(sample_weight)
print('X shape:', X.shape)
print('sample weight summary:', pd.Series(sample_weight).describe().to_dict())


## 2. Fit Constrained and Unconstrained Weights

The optimizer uses analytic gradients. The margin is positive when the recommended video scores above the sampled unrecommended video.


In [ ]:
@dataclass
class FitResult:
    weights: np.ndarray
    loss: float
    initial_loss: float
    converged: bool
    message: str
    num_iterations: int
    gradient_norm: float
    sign_constrained: bool


def logistic_loss_grad(weights, X, sample_weight, tau=1.0, lambda_l2=0.0):
    weights = np.asarray(weights, dtype=np.float64)
    z = (X @ weights) / float(tau)
    # log(1 + exp(-z)) in a stable form.
    loss_terms = np.logaddexp(0.0, -z)
    weight_sum = float(np.sum(sample_weight))
    loss = float(np.sum(sample_weight * loss_terms) / weight_sum + lambda_l2 * float(weights @ weights))
    dloss_dz = -expit(-z)
    grad = (X.T @ (sample_weight * dloss_dz)) / (weight_sum * float(tau)) + 2.0 * lambda_l2 * weights
    return loss, grad


def fit_pairwise_weights(X, sample_weight, sign_constrained=True, tau=1.0, lambda_l2=1e-4, verbose=True):
    start = np.array([1.0, 1.0, 1.0, -1.0], dtype=np.float64) if sign_constrained else np.zeros(4, dtype=np.float64)
    bounds = [(0.0, None), (0.0, None), (0.0, None), (None, 0.0)] if sign_constrained else None
    label = 'constrained' if sign_constrained else 'unconstrained'
    initial_loss = logistic_loss_grad(start, X, sample_weight, tau=tau, lambda_l2=lambda_l2)[0]
    if verbose:
        print(f'[{label}] initial_loss={initial_loss:.8f} start={start.tolist()}')

    iteration = {'n': 0}
    def callback(w):
        iteration['n'] += 1
        if verbose:
            loss, grad = logistic_loss_grad(w, X, sample_weight, tau=tau, lambda_l2=lambda_l2)
            print(f'[{label}] iter={iteration["n"]} loss={loss:.8f} grad_norm={np.linalg.norm(grad):.6g} weights={np.asarray(w).tolist()}', flush=True)

    result = minimize(
        fun=lambda w: logistic_loss_grad(w, X, sample_weight, tau=tau, lambda_l2=lambda_l2)[0],
        jac=lambda w: logistic_loss_grad(w, X, sample_weight, tau=tau, lambda_l2=lambda_l2)[1],
        x0=start,
        method='L-BFGS-B',
        bounds=bounds,
        callback=callback,
        options={'maxiter': 1000, 'ftol': 1e-12, 'gtol': 1e-8},
    )
    final_loss, final_grad = logistic_loss_grad(result.x, X, sample_weight, tau=tau, lambda_l2=lambda_l2)
    if verbose:
        print(f'[{label}] done success={result.success} iterations={result.nit} final_loss={final_loss:.8f}')
    return FitResult(
        weights=result.x.astype(float),
        loss=float(final_loss),
        initial_loss=float(initial_loss),
        converged=bool(result.success),
        message=str(result.message),
        num_iterations=int(result.nit),
        gradient_norm=float(np.linalg.norm(final_grad)),
        sign_constrained=bool(sign_constrained),
    )

fit_constrained = fit_pairwise_weights(X, sample_weight, sign_constrained=True, tau=TAU, lambda_l2=LAMBDA_L2)
fit_unconstrained = fit_pairwise_weights(X, sample_weight, sign_constrained=False, tau=TAU, lambda_l2=LAMBDA_L2)

weights_constrained = fit_constrained.weights
weights_unconstrained = fit_unconstrained.weights
weights_constrained_dict = {name: float(value) for name, value in zip(WEIGHT_NAMES, weights_constrained)}
weights_unconstrained_dict = {name: float(value) for name, value in zip(WEIGHT_NAMES, weights_unconstrained)}

print('constrained weights:')
print(json.dumps(weights_constrained_dict, indent=2))
print('unconstrained weights:')
print(json.dumps(weights_unconstrained_dict, indent=2))


## 3. Metrics and Baselines

For each pair, define the fitted margin

$$
    m_i = w^\top \Delta \hat p_i.
$$

Accuracy is

$$
    \frac{1}{N}\sum_i \mathbf 1\{m_i>0\}.
$$

AUC is computed by making the symmetric binary classification dataset

$$
    \{(m_i,1),(-m_i,0)\}_{i=1}^N.
$$

An AUC above 0.5 means recommended videos tend to receive larger margins than sampled unrecommended videos.


In [ ]:
def metrics_from_margins(margins, sample_weight=None):
    margins = np.asarray(margins, dtype=np.float64)
    if sample_weight is None:
        sample_weight = np.ones_like(margins)
    sample_weight = np.asarray(sample_weight, dtype=np.float64)
    prob = expit(margins / float(TAU))
    labels = np.ones_like(prob)
    labels_auc = np.r_[np.ones_like(margins), np.zeros_like(margins)]
    scores_auc = np.r_[margins / float(TAU), -margins / float(TAU)]
    weights_auc = np.r_[sample_weight, sample_weight]
    return {
        'num_pairs': int(len(margins)),
        'pairwise_accuracy': float(np.average(margins > 0, weights=sample_weight)),
        'pairwise_auc': float(roc_auc_score(labels_auc, scores_auc, sample_weight=weights_auc)),
        'pairwise_log_loss': float(log_loss(labels, prob, labels=[0, 1], sample_weight=sample_weight)),
        'mean_margin': float(np.average(margins, weights=sample_weight)),
        'median_margin': float(np.median(margins)),
    }


def score_with_weights(weights):
    return X @ np.asarray(weights, dtype=np.float64)

baseline_weights = {
    'estimated_constrained': weights_constrained,
    'estimated_unconstrained': weights_unconstrained,
    'single_complete': np.array([1.0, 0.0, 0.0, 0.0]),
    'single_long': np.array([0.0, 1.0, 0.0, 0.0]),
    'single_rewatch': np.array([0.0, 0.0, 1.0, 0.0]),
    'single_neg_negative_weight': np.array([0.0, 0.0, 0.0, -1.0]),
    'equal_signed': np.array([1.0, 1.0, 1.0, -1.0]),
}

comparison = []
for name, weights in baseline_weights.items():
    row = {'model': name}
    row.update(metrics_from_margins(score_with_weights(weights), sample_weight=sample_weight))
    comparison.append(row)
comparison_df = pd.DataFrame(comparison).sort_values('pairwise_auc', ascending=False).reset_index(drop=True)
display(comparison_df)

by_negative_type = []
for neg_type, idx in pairs.groupby('negative_type', observed=True).groups.items():
    idx = np.asarray(list(idx), dtype=np.int64)
    row = {'negative_type': str(neg_type)}
    primary_weights_for_group = weights_constrained if PRIMARY_FIT == 'constrained' else weights_unconstrained
    row.update(metrics_from_margins(X[idx] @ primary_weights_for_group, sample_weight=sample_weight[idx]))
    by_negative_type.append(row)
by_negative_type_df = pd.DataFrame(by_negative_type).sort_values('pairwise_auc', ascending=False).reset_index(drop=True)
display(by_negative_type_df)

delta_summary = pairs[DELTA_COLS].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T.reset_index().rename(columns={'index': 'delta'})
display(delta_summary)


## 4. Save Weights

The top-level `weights` field stores the selected primary fit. Both constrained and unconstrained fits are saved for sensitivity checks.


In [ ]:
primary_weights = weights_constrained if PRIMARY_FIT == 'constrained' else weights_unconstrained
primary_weights_dict = {name: float(value) for name, value in zip(WEIGHT_NAMES, primary_weights)}

optimization_diagnostics = {
    'constrained': {
        **fit_constrained.__dict__,
        'weights': weights_constrained_dict,
        'weight_norm_l1': float(np.abs(weights_constrained).sum()),
        'weight_norm_l2': float(np.linalg.norm(weights_constrained)),
    },
    'unconstrained': {
        **fit_unconstrained.__dict__,
        'weights': weights_unconstrained_dict,
        'weight_norm_l1': float(np.abs(weights_unconstrained).sum()),
        'weight_norm_l2': float(np.linalg.norm(weights_unconstrained)),
    },
}
# Replace ndarray values that came from dataclass __dict__.
optimization_diagnostics['constrained']['weights'] = weights_constrained_dict
optimization_diagnostics['unconstrained']['weights'] = weights_unconstrained_dict

candidate_generation_diagnostics = {}
if CANDIDATE_DIAGNOSTICS_PATH.exists():
    candidate_generation_diagnostics = json.loads(CANDIDATE_DIAGNOSTICS_PATH.read_text())

weights_payload = {
    'weights': primary_weights_dict,
    'primary_fit': PRIMARY_FIT,
    'constrained_weights': weights_constrained_dict,
    'unconstrained_weights': weights_unconstrained_dict,
    'head_order': HEAD_COLS,
    'delta_order': DELTA_COLS,
    'weight_order': WEIGHT_NAMES,
    'score_formula': 'w_complete*p_complete + w_long*p_long + w_rewatch*p_rewatch + w_neg*p_neg',
    'objective': {
        'name': 'recommended-vs-sampled-unrecommended Bradley-Terry pairwise logistic loss',
        'convex': True,
        'temperature': TAU,
        'lambda_l2': LAMBDA_L2,
        'pair_weighting': PAIR_WEIGHTING,
        'primary_fit': PRIMARY_FIT,
        'sign_constraints_for_constrained_fit': {
            'w_complete': '[0, +inf)',
            'w_long': '[0, +inf)',
            'w_rewatch': '[0, +inf)',
            'w_neg': '(-inf, 0]',
        },
    },
    'sample_summary': sample_summary,
    'optimization_diagnostics': optimization_diagnostics,
    'candidate_generation_diagnostics': candidate_generation_diagnostics,
}

metrics_payload = {
    'sample_summary': sample_summary,
    'comparison': comparison_df.to_dict(orient='records'),
    'by_negative_type': by_negative_type_df.to_dict(orient='records'),
    'delta_summary': delta_summary.to_dict(orient='records'),
    'optimization_diagnostics': optimization_diagnostics,
    'candidate_generation_diagnostics': candidate_generation_diagnostics,
}

scored = pairs[[
    'pair_id', 'session_key', 'user_id', 'session', 'burst_id', 'session_start_time',
    'j_plus', 'j_minus', 'negative_type', 'negative_exclusion_policy',
    'eligible_pool_size', 'negative_prior_seen_by_user', 'negative_future_seen_by_user', 'negative_ever_seen_by_user',
    *DELTA_COLS,
]].copy()
for name, weights in baseline_weights.items():
    scored[f'margin_{name}'] = X @ weights
scored['margin_primary'] = X @ primary_weights

report = f"""# Head-Weight Pairwise Estimation Report

Input pair deltas: `{PAIR_DELTAS_PATH}`

## Sample

- Pairs: {sample_summary['pairs']:,}
- Sessions: {sample_summary['sessions']:,}
- Users: {sample_summary['users']:,}
- Positive session-videos: {sample_summary['positive_session_videos']:,}
- Negative session-videos: {sample_summary['negative_session_videos']:,}
- Negative exclusion policies: {sample_summary['negative_exclusion_policies']}
- Future same-user recommendations sampled as negatives: {sample_summary['negative_future_seen_by_user_sum']}
- Ever same-user recommendations sampled as negatives: {sample_summary['negative_ever_seen_by_user_sum']}

## Objective

Convex Bradley-Terry pairwise logistic loss with L2 penalty.

- temperature: {TAU}
- lambda_l2: {LAMBDA_L2}
- pair weighting: `{PAIR_WEIGHTING}`
- primary fit: `{PRIMARY_FIT}`

## Primary Weights

```json
{json.dumps(primary_weights_dict, indent=2)}
```

## Comparison

```json
{json.dumps(comparison_df.to_dict(orient='records'), indent=2)}
```

## Score Formula

$$
\\operatorname{{score}}
= w_{{\text{{complete}}}}p_{{\text{{complete}}}}
+ w_{{\text{{long}}}}p_{{\text{{long}}}}
+ w_{{\text{{rewatch}}}}p_{{\text{{rewatch}}}}
+ w_{{\text{{neg}}}}p_{{\text{{neg}}}}.
$$
"""

if WRITE_OUTPUTS:
    WEIGHTS_PATH.write_text(json.dumps(weights_payload, indent=2, default=str) + '\n')
    METRICS_PATH.write_text(json.dumps(metrics_payload, indent=2, default=str) + '\n')
    scored.to_parquet(SCORES_PATH, index=False)
    REPORT_PATH.write_text(report)
    print('saved weights:', WEIGHTS_PATH)
    print('saved metrics:', METRICS_PATH)
    print('saved scored pairs:', SCORES_PATH)
    print('saved report:', REPORT_PATH)
else:
    print('WRITE_OUTPUTS is False; skipped saving.')

print(json.dumps(primary_weights_dict, indent=2))
